# Conser-vision — classer les animaux des pièges photographiques

**Compétition :** DrivenData *Conser-vision Practice Area: Image Classification*
**Tâche :** pour chaque photo de piège photographique du parc national de Taï (Côte d'Ivoire), donner la probabilité de chacune des 8 classes : `antelope_duiker`, `bird`, `blank` (aucun animal), `civet_genet`, `hog`, `leopard`, `monkey_prosimian`, `rodent`.
**Métrique :** *log loss* multi-classe (plus c'est bas, mieux c'est). Repères du classement : n°1 ≈ 0,53 · 10<sup>e</sup> ≈ 0,63 · 20<sup>e</sup> ≈ 0,80 · 50<sup>e</sup> ≈ 1,02.

### Le plan
| Étape | Ce qu'on fait | Durée sur T4 (ordre de grandeur) |
|---|---|---|
| 1–2 | Installation, téléchargement des données (une seule fois, sauvegardées sur Drive) | 5–15 min la 1<sup>re</sup> fois |
| 3 | Pré-redimensionner les images (accélère tout le reste) | ~5 min, une seule fois |
| 4 | Explorer les données | 2 min |
| 5 | Validation croisée **par site** (le point clé de cette compétition) | instantané |
| 6 | **Modèle A** : embeddings de modèles pré-entraînés (DINOv2, CLIP) + régression logistique → 1<sup>re</sup> soumission | ~15 min |
| 7 | **Modèle B** : fine-tuning d'un ConvNeXt, fold par fold | ~15–20 min par fold (×5) |
| 8 | Mélange des modèles + calibration (température) → soumission finale | 2 min |

### Comment reprendre après une déconnexion
Colab gratuit coupe les sessions au bout de quelques heures. **Tout ce qui est long est sauvegardé sur Google Drive** (données, embeddings, prédictions de chaque fold). Si la session s'arrête : *Exécution → Tout exécuter*. Les étapes déjà faites sont détectées et sautées ; l'entraînement reprend au premier fold non terminé.

**Avant de commencer :** *Exécution → Modifier le type d'exécution → GPU T4*.

**Règlement :** pas de données externes ; les modèles pré-entraînés publics sont autorisés (on utilise ImageNet, DINOv2 et CLIP). Les données ne doivent pas être redistribuées : ne publie jamais les images ni les CSV.

---
## 1. Installation et réglages

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"   # réservé aux tests automatiques (petites fausses données sur CPU)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")

import glob, json, math, shutil, time, zipfile, warnings
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as Tr
import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU :", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "AUCUN — active le GPU (Exécution → Modifier le type d'exécution)")
print("torch", torch.__version__, "| timm", timm.__version__)

In [ ]:
# ⚙️ Réglages
DRIVE_DIR = "/content/drive/MyDrive/conservision" if IN_COLAB else os.path.abspath("cv_drive")   # survit aux sessions
LOCAL_DIR = "/content/conservision" if IN_COLAB else os.path.abspath("cv_local")                 # disque rapide de la session
SEED, N_FOLDS = 42, 5
CLASSES = ["antelope_duiker", "bird", "blank", "civet_genet", "hog", "leopard", "monkey_prosimian", "rodent"]

SMALL_LONG_SIDE = 640          # étape 3 : on stocke les images avec un grand côté de 640 px

# Modèle A : extracteurs d'embeddings (gelés) + régression logistique
EMB_MODELS = {
    "dinov2_b": dict(name="vit_base_patch14_dinov2.lvd142m", img=224, pretrained=True),
    "clip_l":   dict(name="vit_large_patch14_clip_224.openai", img=224, pretrained=True),
}
# Modèle B : fine-tuning
FT = dict(tag="convnext_tiny_288x512", name="convnext_tiny.fb_in22k_ft_in1k", pretrained=True,
          img=(288, 512), epochs=10, batch=32, lr=2e-4, wd=0.05, drop_path=0.1)
NUM_WORKERS = min(4, os.cpu_count() or 2)

if SMOKE_TEST:   # versions minuscules pour vérifier que tout s'exécute sans GPU ni téléchargement
    SMALL_LONG_SIDE = 160
    EMB_MODELS = {"tiny_vit": dict(name="vit_tiny_patch16_224", img=64, pretrained=False)}
    FT = dict(tag="smoke_resnet", name="resnet18", pretrained=False, img=(64, 112), epochs=2, batch=16,
              lr=1e-3, wd=0.0, drop_path=0.0)

os.makedirs(DRIVE_DIR, exist_ok=True); os.makedirs(LOCAL_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
print("Drive :", DRIVE_DIR, "| local :", LOCAL_DIR)

---
## 2. Les données (téléchargées une seule fois, puis gardées sur Drive)

La première fois, la cellule te demande un **lien de téléchargement** :
1. Va sur la page *Data download* de la compétition (connectée à ton compte DrivenData).
2. Fais un **clic droit sur `competition.zip` → Copier l'adresse du lien**.
3. Colle le lien dans la case qui apparaît sous la cellule, puis Entrée.

Le lien est personnel et expire au bout de 24 h : ne le partage pas. L'archive est copiée sur ton Drive (`MyDrive/conservision/`), donc les sessions suivantes n'ont plus besoin du lien.

In [ ]:
ZIP_DRIVE = os.path.join(DRIVE_DIR, "competition.zip")
ZIP_LOCAL = os.path.join(LOCAL_DIR, "competition.zip")
RAW_DIR = os.path.join(LOCAL_DIR, "raw")


def download(url, dest):
    import urllib.request
    with urllib.request.urlopen(url) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length", 0)); done = 0; t0 = time.time()
        while chunk := r.read(1 << 22):
            f.write(chunk); done += len(chunk)
            if total:
                print(f"\r{done / 1e9:.2f} / {total / 1e9:.2f} Go  ({done / max(time.time() - t0, 1e-9) / 1e6:.0f} Mo/s)", end="")
    print()


if not glob.glob(os.path.join(RAW_DIR, "**", "train_labels.csv"), recursive=True):
    if not os.path.exists(ZIP_DRIVE):
        url = input("Colle le lien de competition.zip : ").strip()
        download(url, ZIP_LOCAL)
        print("Copie vers Drive…"); shutil.copy(ZIP_LOCAL, ZIP_DRIVE)
    elif not os.path.exists(ZIP_LOCAL):
        print("Copie depuis Drive…"); shutil.copy(ZIP_DRIVE, ZIP_LOCAL)
    print("Décompression…")
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(RAW_DIR)
    os.remove(ZIP_LOCAL)   # la copie sur Drive suffit

DATA = os.path.dirname(glob.glob(os.path.join(RAW_DIR, "**", "train_labels.csv"), recursive=True)[0])
print("Données dans :", DATA)
print(sorted(os.listdir(DATA)))

In [ ]:
train = pd.read_csv(f"{DATA}/train_features.csv", index_col="id")
labels = pd.read_csv(f"{DATA}/train_labels.csv", index_col="id")
test = pd.read_csv(f"{DATA}/test_features.csv", index_col="id")
sub_fmt = pd.read_csv(f"{DATA}/submission_format.csv", index_col="id")

assert (train.index == labels.index).all() and (test.index == sub_fmt.index).all()
assert list(labels.columns) == CLASSES == list(sub_fmt.columns)
y = labels.values.argmax(axis=1)          # classe sous forme d'entier 0..7
print(f"{len(train)} images d'entraînement, {len(test)} images de test")
print(f"{train.site.nunique()} sites d'entraînement, {test.site.nunique()} sites de test, "
      f"sites communs : {len(set(train.site) & set(test.site))}")
train.head()

---
## 3. Pré-redimensionner les images (une seule fois)

Les photos originales sont grandes ; les décompresser à chaque époque d'entraînement serait le goulot d'étranglement (Colab n'a que 2 cœurs CPU). On les réduit **une fois** à 640 px de grand côté, en gardant les proportions. L'astuce `Image.draft` demande au décodeur JPEG de décoder directement en plus petit, ce qui est beaucoup plus rapide.

Le résultat est zippé sur Drive pour les sessions suivantes.

In [ ]:
SMALL_DIR = os.path.join(LOCAL_DIR, f"small{SMALL_LONG_SIDE}")
SMALL_ZIP = os.path.join(DRIVE_DIR, f"small{SMALL_LONG_SIDE}.zip")


def shrink(rel):
    src, dst = os.path.join(DATA, rel), os.path.join(SMALL_DIR, rel)
    if os.path.exists(dst):
        return
    im = Image.open(src); im.draft("RGB", (SMALL_LONG_SIDE, SMALL_LONG_SIDE)); im = im.convert("RGB")
    im.thumbnail((SMALL_LONG_SIDE, SMALL_LONG_SIDE), Image.BILINEAR)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    im.save(dst, quality=93)


all_paths = list(train.filepath) + list(test.filepath)
if not os.path.isdir(SMALL_DIR) and os.path.exists(SMALL_ZIP):
    print("Récupération des images réduites depuis Drive…")
    with zipfile.ZipFile(SMALL_ZIP) as z:
        z.extractall(SMALL_DIR)
todo = [p for p in all_paths if not os.path.exists(os.path.join(SMALL_DIR, p))]
if todo:
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=8) as ex:
        for i, _ in enumerate(ex.map(shrink, todo)):
            if i % 2000 == 0:
                print(f"{i}/{len(todo)}  ({time.time() - t0:.0f} s)")
    print(f"{len(todo)} images réduites en {time.time() - t0:.0f} s — sauvegarde sur Drive…")
    tmp = shutil.make_archive(os.path.join(LOCAL_DIR, "small_tmp"), "zip", SMALL_DIR)   # zip local, puis copie
    shutil.copy(tmp, SMALL_ZIP); os.remove(tmp)
print("Images réduites prêtes :", SMALL_DIR)

---
## 4. Explorer les données

In [ ]:
counts = labels.sum().sort_values()
counts.plot.barh(figsize=(7, 3.5), title="Nombre d'images par classe"); plt.show()
print((counts / counts.sum()).round(3).sort_values(ascending=False).to_string())

In [ ]:
# Un exemple par classe (les images de nuit sont en noir et blanc : infrarouge)
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
rng = np.random.default_rng(0)
for ax, c in zip(axes.ravel(), CLASSES):
    rid = rng.choice(labels.index[labels[c] == 1])
    ax.imshow(Image.open(os.path.join(SMALL_DIR, train.loc[rid, "filepath"]))); ax.set_title(f"{c}\n{rid}"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Tailles des images d'origine et part d'images de nuit (sur un échantillon)
sample = train.sample(min(400, len(train)), random_state=0)
sizes = pd.Series([Image.open(os.path.join(DATA, p)).size for p in sample.filepath]).value_counts()
print("Tailles d'origine les plus fréquentes (largeur, hauteur) :\n", sizes.head(5).to_string())


def is_gray(p):
    a = np.asarray(Image.open(os.path.join(SMALL_DIR, p)).convert("RGB").resize((64, 36)), dtype=np.int16)
    return np.abs(a[..., 0] - a[..., 1]).mean() < 2 and np.abs(a[..., 1] - a[..., 2]).mean() < 2


sample["nuit"] = [is_gray(p) for p in sample.filepath]
print(f"\nPart d'images infrarouges (noir et blanc) : {sample.nuit.mean():.0%}")
print(sample.assign(classe=labels.loc[sample.index].idxmax(axis=1)).groupby("classe").nuit.mean().round(2).to_string())

In [ ]:
# Les sites : combien d'images, et la répartition des classes varie-t-elle d'un site à l'autre ?
per_site = train.site.value_counts()
print(f"Images par site : min {per_site.min()}, médiane {int(per_site.median())}, max {per_site.max()}")
mix = labels.groupby(train.site).mean()
top = per_site.index[:15]
plt.figure(figsize=(10, 4)); plt.imshow(mix.loc[top].T, aspect="auto", cmap="viridis")
plt.yticks(range(len(CLASSES)), CLASSES); plt.xticks(range(len(top)), top, rotation=90)
plt.colorbar(label="part des images du site"); plt.title("Répartition des classes dans les 15 plus gros sites"); plt.show()

**🧠 À comprendre.** Chaque caméra filme toujours le même décor, et certains animaux fréquentent surtout certains sites. Un modèle peut donc « tricher » : reconnaître le **décor** (cet arbre, cet éclairage) au lieu de l'**animal**. Comme les sites du test sont **tous nouveaux**, cette triche ne marche pas au moment de la soumission. C'est pour ça que la validation doit séparer les sites (étape 5).

---
## 5. La validation croisée par site

On utilise `StratifiedGroupKFold` :
- **Group** : toutes les images d'un même site vont dans le même fold, donc on valide toujours sur des sites jamais vus, exactement comme le test ;
- **Stratified** : on essaie de garder une répartition des classes comparable d'un fold à l'autre.

Si on faisait un découpage aléatoire classique, des images du même site (presque identiques) seraient à la fois en entraînement et en validation, et le score de validation serait **beaucoup trop optimiste**.

Le découpage est sauvegardé sur Drive : tous les modèles utilisent exactement les mêmes folds, ce qui permet de les comparer et de les mélanger.

In [ ]:
FOLDS_CSV = os.path.join(DRIVE_DIR, "folds.csv")
if os.path.exists(FOLDS_CSV):
    fold = pd.read_csv(FOLDS_CSV, index_col="id").loc[train.index, "fold"].values
else:
    fold = np.zeros(len(train), dtype=int)
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for k, (_, va) in enumerate(sgkf.split(train, y, groups=train.site)):
        fold[va] = k
    pd.DataFrame({"id": train.index, "fold": fold}).to_csv(FOLDS_CSV, index=False)

for k in range(N_FOLDS):   # vérification : aucun site n'est partagé entre entraînement et validation
    assert not set(train.site[fold == k]) & set(train.site[fold != k])
print(pd.crosstab(fold, labels.idxmax(axis=1), normalize="index").round(2))
print("\nImages par fold :", np.bincount(fold))

In [ ]:
def logloss(y_int, prob):
    """Log loss multi-classe, comme sur le classement (probabilités bornées pour éviter log(0))."""
    p = np.clip(prob, 1e-15, 1 - 1e-15); p = p / p.sum(axis=1, keepdims=True)
    return log_loss(y_int, p, labels=list(range(len(CLASSES))))


def fit_temperature(logp, y_int):
    """Calibration : trouve T tel que softmax(logp / T) minimise la log loss (sur les OOF)."""
    res = minimize_scalar(lambda t: logloss(y_int, softmax(logp / t, axis=1)), bounds=(0.3, 5), method="bounded")
    return res.x


def report(name, prob_oof):
    ok = ~np.isnan(prob_oof).any(axis=1)
    print(f"{name:28s} log loss OOF = {logloss(y[ok], prob_oof[ok]):.4f} | accuracy = "
          f"{accuracy_score(y[ok], prob_oof[ok].argmax(1)):.3f} | {ok.mean():.0%} des images couvertes")


# Repère : prédire toujours la répartition des classes de l'entraînement
prior = np.tile(labels.mean().values, (len(train), 1))
report("Référence (fréquences)", prior)

**🧠 La log loss en deux phrases.** Pour chaque image, on regarde la probabilité donnée à la **bonne** classe et on prend −log de cette probabilité, puis on fait la moyenne. Être sûr et avoir raison coûte presque 0 ; être sûr et se tromper (p = 0,01 sur la bonne classe) coûte 4,6. La log loss punit donc surtout l'**excès de confiance** : un modèle doit savoir dire « je ne suis pas sûr ».

---
## 6. Modèle A — embeddings pré-entraînés + régression logistique

Idée : un grand modèle de vision déjà entraîné sur des millions d'images (DINOv2, CLIP) sait transformer une photo en un vecteur de nombres (un **embedding**) qui résume son contenu. On ne ré-entraîne pas ce modèle : on calcule l'embedding de chaque image **une fois**, puis on entraîne une simple **régression logistique** dessus.

- C'est rapide (quelques minutes) et très robuste aux nouveaux sites, car le modèle n'a pas pu apprendre nos décors.
- C'est notre **première soumission** et notre point de comparaison pour le fine-tuning.

On fait aussi une **TTA** (*test-time augmentation*) : on calcule l'embedding de l'image et de son miroir horizontal, et on fait la moyenne.

In [ ]:
class ImgDS(Dataset):
    def __init__(self, paths, tf, labels=None):
        self.paths, self.tf, self.labels = list(paths), tf, labels
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        im = Image.open(os.path.join(SMALL_DIR, self.paths[i])).convert("RGB")
        x = self.tf(im)
        return (x, self.labels[i]) if self.labels is not None else x


def loader(ds, bs, shuffle=False, drop_last=False):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=drop_last, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE == "cuda", persistent_workers=NUM_WORKERS > 0)


@torch.no_grad()
def embed(cfg, paths):
    m = timm.create_model(cfg["name"], pretrained=cfg["pretrained"], num_classes=0, img_size=cfg["img"]).to(DEVICE).eval()
    dc = m.pretrained_cfg
    tf = Tr.Compose([Tr.Resize((cfg["img"], cfg["img"])), Tr.ToTensor(), Tr.Normalize(dc["mean"], dc["std"])])
    out = []
    for x in loader(ImgDS(paths, tf), 64):
        x = x.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            e = (m(x) + m(torch.flip(x, dims=[3]))) / 2      # TTA : image + miroir
        out.append(e.float().cpu().numpy())
    del m; torch.cuda.empty_cache()
    return np.concatenate(out)


EMB = {}
for key, cfg in EMB_MODELS.items():
    f = os.path.join(DRIVE_DIR, f"emb_{key}.npz")
    if not os.path.exists(f):
        t0 = time.time()
        np.savez(f, train=embed(cfg, train.filepath), test=embed(cfg, test.filepath))
        print(f"{key} : embeddings calculés en {time.time() - t0:.0f} s")
    d = np.load(f); EMB[key] = (d["train"], d["test"])
    print(f"{key:10s} dimension {EMB[key][0].shape[1]}")

In [ ]:
def standardize(Etr, Ete):
    """Centre-réduit chaque dimension avec les statistiques de l'entraînement (pas d'étiquettes utilisées)."""
    mu, sd = Etr.mean(0), Etr.std(0) + 1e-6
    return (Etr - mu) / sd, (Ete - mu) / sd


def logreg_cv(Xtr, Xte, C):
    oof, te = np.zeros((len(Xtr), len(CLASSES))), np.zeros((len(Xte), len(CLASSES)))
    for k in range(N_FOLDS):
        m = LogisticRegression(C=C, max_iter=500, tol=1e-3)
        m.fit(Xtr[fold != k], y[fold != k])
        oof[fold == k] = m.predict_proba(Xtr[fold == k])
        te += m.predict_proba(Xte) / N_FOLDS
    return oof, te


FEATS = {k: standardize(*v) for k, v in EMB.items()}
if len(EMB) > 1:   # bout à bout : les deux modèles réunis
    FEATS["concat"] = tuple(np.hstack([FEATS[k][i] for k in EMB]) for i in (0, 1))

PRED = {}   # PRED[nom] = (probabilités OOF, probabilités test)
for key, (Etr, Ete) in FEATS.items():
    best = None
    for C in [0.003, 0.01, 0.03, 0.1]:
        t0 = time.time(); oof, te = logreg_cv(Etr, Ete, C); s = logloss(y, oof)
        print(f"  {key:10s} C={C:<6} log loss OOF = {s:.4f}  ({time.time() - t0:.0f} s)")
        if best is None or s < best[0]:
            best = (s, C, oof, te)
    PRED[f"emb_{key}"] = (best[2], best[3])
    report(f"emb_{key} (C={best[1]})", best[2])

**🧠 À comprendre.** On choisit `C` (l'inverse de la force de régularisation) d'après la log loss OOF. Avec seulement 4 valeurs testées, le risque de sur-ajuster ce choix est faible. La ligne `concat` met bout à bout les embeddings des deux modèles : s'ils capturent des informations différentes, la combinaison fait mieux que chacun.

### Première soumission
On prend le meilleur modèle A et on le **calibre** avec une température avant d'écrire la soumission (voir l'étape 8 pour l'explication).

In [ ]:
def make_submission(prob_test, name):
    sub = sub_fmt.copy()
    sub[CLASSES] = prob_test / prob_test.sum(axis=1, keepdims=True)
    assert sub.shape == sub_fmt.shape and list(sub.columns) == CLASSES and (sub.index == sub_fmt.index).all()
    assert not sub.isna().any().any() and np.allclose(sub.sum(axis=1), 1) and (sub.values >= 0).all()
    path = os.path.join(DRIVE_DIR, name); sub.to_csv(path)
    print("Soumission écrite :", path)
    if IN_COLAB:
        from google.colab import files
        files.download(path)
    return sub


best_A = min([k for k in PRED if k.startswith("emb_")], key=lambda k: logloss(y, PRED[k][0]))
oofA, teA = PRED[best_A]
T_A = fit_temperature(np.log(np.clip(oofA, 1e-15, 1)), y)
print(f"Meilleur modèle A : {best_A} — température {T_A:.2f} : log loss OOF {logloss(y, oofA):.4f} → "
      f"{logloss(y, softmax(np.log(np.clip(oofA, 1e-15, 1)) / T_A, axis=1)):.4f}")
_ = make_submission(softmax(np.log(np.clip(teA, 1e-15, 1)) / T_A, axis=1), "submission_A.csv")

---
## 7. Modèle B — fine-tuning d'un ConvNeXt

Cette fois, on ré-entraîne **tout** le réseau (pré-entraîné sur ImageNet-22k) pour notre tâche. C'est plus long mais en général bien meilleur, car le réseau apprend à repérer de petits animaux, de nuit, partiellement cachés…

**Les choix et pourquoi :**
- **Résolution 288 × 512** (format paysage, comme les photos) : les animaux sont souvent petits dans l'image ; une résolution plus haute aide.
- **Augmentations** : recadrage aléatoire, miroir, variations de couleur et passage aléatoire en noir et blanc (pour imiter les images de nuit). Le modèle voit des variantes de chaque photo et généralise mieux aux nouveaux sites.
- **AdamW + schedule cosinus avec warmup** : le taux d'apprentissage monte doucement pendant la première époque (pour ne pas abîmer les poids pré-entraînés), puis décroît.
- **Précision mixte (AMP)** : calculs en float16 sur le GPU, presque 2× plus rapide.
- **Meilleure époque selon la log loss de validation** : on garde les poids de l'époque où le modèle était le meilleur sur les sites jamais vus.

Chaque fold terminé est sauvegardé sur Drive (prédictions OOF et test). Si la session coupe, relance tout : les folds faits sont sautés.

In [ ]:
FT_DIR = os.path.join(DRIVE_DIR, f"ft_{FT['tag']}"); os.makedirs(FT_DIR, exist_ok=True)
H, W = FT["img"]
_cfg = timm.create_model(FT["name"], pretrained=False).pretrained_cfg   # moyenne / écart-type de normalisation du modèle
MEAN, STD = _cfg.get("mean", (0.485, 0.456, 0.406)), _cfg.get("std", (0.229, 0.224, 0.225))

tf_train = Tr.Compose([
    Tr.RandomResizedCrop((H, W), scale=(0.6, 1.0), ratio=(W / H * 0.8, W / H * 1.25)),
    Tr.RandomHorizontalFlip(),
    Tr.ColorJitter(0.3, 0.3, 0.2),
    Tr.RandomGrayscale(p=0.15),
    Tr.ToTensor(), Tr.Normalize(MEAN, STD)])
tf_eval = Tr.Compose([Tr.Resize((H, W)), Tr.ToTensor(), Tr.Normalize(MEAN, STD)])


@torch.no_grad()
def predict_logits(model, paths):
    model.eval(); out = []
    for x in loader(ImgDS(paths, tf_eval), FT["batch"] * 2):
        x = x.to(DEVICE).to(memory_format=torch.channels_last)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            p = (model(x).float().softmax(1) + model(torch.flip(x, dims=[3])).float().softmax(1)) / 2  # TTA : image + miroir
        out.append(torch.log(p.clamp_min(1e-15)).cpu().numpy())
    return np.concatenate(out)       # log-probabilités


def train_fold(k):
    done = os.path.join(FT_DIR, f"fold{k}.npz")
    if os.path.exists(done):
        print(f"Fold {k} déjà fait ✔"); return
    tr_idx, va_idx = np.where(fold != k)[0], np.where(fold == k)[0]
    model = timm.create_model(FT["name"], pretrained=FT["pretrained"], num_classes=len(CLASSES),
                              drop_path_rate=FT["drop_path"]).to(DEVICE).to(memory_format=torch.channels_last)
    dl = loader(ImgDS(train.filepath.values[tr_idx], tf_train, y[tr_idx]), FT["batch"], shuffle=True, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=FT["lr"], weight_decay=FT["wd"])
    total = FT["epochs"] * len(dl); warm = len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1, s / total))))
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    best, best_state = 9e9, None
    for ep in range(FT["epochs"]):
        model.train(); t0 = time.time(); run = 0.0
        for x, t in dl:
            x, t = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last), t.to(DEVICE)
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                loss = F.cross_entropy(model(x), t)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
        val_lp = predict_logits(model, train.filepath.values[va_idx])
        s = logloss(y[va_idx], np.exp(val_lp))
        print(f"fold {k} époque {ep + 1}/{FT['epochs']} — perte train {run / len(dl):.3f} | log loss val {s:.4f} "
              f"| acc val {accuracy_score(y[va_idx], val_lp.argmax(1)):.3f} | {time.time() - t0:.0f} s")
        if s < best:
            best, best_state = s, {n: p.detach().cpu().clone() for n, p in model.state_dict().items()}
    model.load_state_dict(best_state)
    np.savez(done, va_idx=va_idx, val=predict_logits(model, train.filepath.values[va_idx]),
             test=predict_logits(model, test.filepath.values), best=best)
    print(f"✔ Fold {k} terminé — meilleure log loss val {best:.4f}, sauvegardé sur Drive")
    del model, opt; torch.cuda.empty_cache()

In [ ]:
# Lance (ou reprend) l'entraînement. Tu peux limiter les folds de cette session, ex. FOLDS_A_FAIRE = [0, 1]
FOLDS_A_FAIRE = list(range(N_FOLDS))
for k in FOLDS_A_FAIRE:
    train_fold(k)

In [ ]:
def load_ft(tag_dir):
    oof = np.full((len(train), len(CLASSES)), np.nan); te = []
    for k in range(N_FOLDS):
        f = os.path.join(tag_dir, f"fold{k}.npz")
        if os.path.exists(f):
            d = np.load(f); oof[d["va_idx"]] = np.exp(d["val"]); te.append(np.exp(d["test"]))
    return oof, (np.mean(te, axis=0) if te else None), len(te)


oofB, teB, nB = load_ft(FT_DIR)
print(f"{nB}/{N_FOLDS} folds disponibles")
if nB:
    report(f"ft_{FT['tag']}", oofB)
    if nB == N_FOLDS:
        PRED[f"ft_{FT['tag']}"] = (oofB, teB)

**🧠 À lire dans les logs.** Si la perte d'entraînement continue de baisser alors que la log loss de validation remonte, le modèle sur-apprend (il mémorise les sites d'entraînement). C'est pour ça qu'on garde la **meilleure époque**, pas la dernière.

---
## 8. Mélange + calibration → soumission finale

1. **Mélange** : moyenne pondérée des probabilités des modèles A et B. Le poids est choisi sur les prédictions OOF (un seul paramètre, donc peu de risque de sur-ajustement). Si seuls certains folds de B sont terminés, le poids et la température sont choisis sur les images de ces folds, et les prédictions test de B sont la moyenne des folds disponibles.
2. **Calibration par température** : on divise les log-probabilités par un nombre T avant le softmax. T > 1 rend le modèle moins sûr de lui, T < 1 plus sûr. On choisit T qui minimise la log loss OOF. Comme la log loss punit l'excès de confiance, ce petit réglage fait souvent gagner quelques centièmes.

In [ ]:
best_emb = min([k for k in PRED if k.startswith("emb_")], key=lambda k: logloss(y, PRED[k][0]))
oofA, teA = PRED[best_emb]
lp = lambda p: np.log(np.clip(p, 1e-15, 1))
if nB:   # au moins un fold de B : poids et température choisis sur les images couvertes par ces folds
    cal = ~np.isnan(oofB).any(axis=1)
    grid = np.linspace(0, 1, 21)
    scores = [logloss(y[cal], w * oofB[cal] + (1 - w) * oofA[cal]) for w in grid]
    w = grid[int(np.argmin(scores))]
    print(f"Modèle B : {nB}/{N_FOLDS} folds, {cal.mean():.0%} des images. Sur ces images : A {scores[0]:.4f} | "
          f"B {scores[-1]:.4f} | mélange {min(scores):.4f} (poids de B = {w:.2f})")
    plt.plot(grid, scores); plt.xlabel("poids du modèle B"); plt.ylabel("log loss OOF"); plt.show()
    oof_mix, te_mix = w * oofB[cal] + (1 - w) * oofA[cal], w * teB + (1 - w) * teA
else:
    print("Pas encore de modèle B : on utilise le meilleur modèle A.")
    cal = np.ones(len(train), dtype=bool)
    oof_mix, te_mix = oofA, teA

T = fit_temperature(lp(oof_mix), y[cal])
oof_final, te_final = softmax(lp(oof_mix) / T, axis=1), softmax(lp(te_mix) / T, axis=1)
print(f"Température T = {T:.2f} : log loss OOF {logloss(y[cal], oof_mix):.4f} → {logloss(y[cal], oof_final):.4f} "
      f"| accuracy {accuracy_score(y[cal], oof_final.argmax(1)):.3f} (sur {cal.mean():.0%} des images)")
sub = make_submission(te_final, "submission_final.csv")
sub.head()

In [ ]:
# Où le modèle se trompe-t-il ? Matrice de confusion OOF (en proportion de chaque vraie classe)
cm = pd.crosstab(pd.Series(np.array(CLASSES)[y[cal]], name="vraie"), pd.Series(np.array(CLASSES)[oof_final.argmax(1)], name="prédite"), normalize="index")
plt.figure(figsize=(7, 6)); plt.imshow(cm.values, cmap="Blues", vmin=0, vmax=1)
plt.xticks(range(len(cm.columns)), cm.columns, rotation=60, ha="right"); plt.yticks(range(len(cm.index)), cm.index)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, f"{cm.values[i, j]:.2f}", ha="center", va="center", color="white" if cm.values[i, j] > 0.5 else "black", fontsize=8)
plt.title("Confusion OOF (lignes = vraie classe)"); plt.tight_layout(); plt.show()

# Le score OOF par fold donne une idée de la variabilité d'un site à l'autre
fc = fold[cal]
print(pd.Series({k: logloss(y[cal][fc == k], oof_final[fc == k]) for k in range(N_FOLDS) if (fc == k).any()}, name="log loss par fold").round(4))

---
## 9. Et ensuite ?

Note tes scores (OOF et classement public) dans un tableau : c'est ce qui te permettra de raconter la démarche.

Pistes d'amélioration, de la plus simple à la plus coûteuse :
1. **Plus d'époques** (`epochs=15`) ou un taux d'apprentissage différent (`lr=1e-4`, `3e-4`) pour le fine-tuning. Change le `tag` dans `FT` pour ne pas écraser les résultats précédents.
2. **Un modèle plus gros** : `convnext_small.fb_in22k_ft_in1k` (≈ 2× plus lent).
3. **Plusieurs modèles fine-tunés** dans le mélange (ex. ConvNeXt + EfficientNetV2), chacun avec son `tag`.
4. **Une résolution plus haute** (`img=(384, 672)`) : les petits animaux (rongeurs, oiseaux) en profitent, mais c'est ≈ 1,8× plus lent.

Avant toute idée qui utilise les images du test pour l'entraînement (pseudo-labeling), vérifie sur le forum de la compétition que c'est autorisé.